# Complete AI-PID-Automation Notebook

Complete cleaned notebook containing all modules.


In [ ]:
import numpy as np

def find_tag_xy(df_text, tag):
    pat = rf"\b{tag.replace('-','[-\\s]?')}\b"
    hits = df_text[df_text["text"].astype(str).str.contains(pat, case=False, regex=True)]
    if len(hits) == 0:
        raise ValueError(f"Tag not found: {tag}")
    return hits.iloc[0][["x","y"]].values.astype(float)

def nearest_insert_index(df_main, xy):
    pts = df_main[["x","y"]].values.astype(float)
    d = np.sqrt(((pts - xy)**2).sum(axis=1))
    return df_main.iloc[np.argmin(d)].name

reactor_tags = ["R-100","R-200","R-300"]
reactor_nodes = {}

for t in reactor_tags:
    xy = find_tag_xy(df_text, t)
    ins_idx = nearest_insert_index(df_main, xy)
    reactor_nodes[t] = f"ins_{ins_idx}"
    # stamp role
    G.nodes[reactor_nodes[t]]["unit_tag"] = t
    G.nodes[reactor_nodes[t]]["unit_type"] = "reactor"

G.nodes[reactor_nodes["R-300"]]["reactor_stage"] = "catalytic_fixed_bed"
G.nodes[reactor_nodes["R-100"]]["reactor_stage"] = "acidification"
G.nodes[reactor_nodes["R-200"]]["reactor_stage"] = "acidification"

reactor_nodes

In [ ]:
!pip -q install ezdxf shapely networkx pandas matplotlib
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install torch-geometric -q


In [ ]:
from google.colab import files
uploaded = files.upload()   # upload your .dxf
DXF_FILE = [k for k in uploaded.keys() if k.lower().endswith(".dxf")][0]
DXF_FILE


In [ ]:
import ezdxf
import pandas as pd
import numpy as np

doc = ezdxf.readfile(DXF_FILE)
msp = doc.modelspace()

# 1A) TEXT / MTEXT
texts = []
for e in msp.query("TEXT MTEXT"):
    if e.dxftype() == "TEXT":
        s = e.dxf.text
        x, y = e.dxf.insert.x, e.dxf.insert.y
        layer = e.dxf.layer
    else:
        s = e.text
        x, y = e.dxf.insert.x, e.dxf.insert.y
        layer = e.dxf.layer
    texts.append({"text": str(s).strip(), "x": x, "y": y, "layer": layer})

df_text = pd.DataFrame(texts)

# 1B) INSERT (Blocks: valves/instruments/equipment symbols)
inserts = []
for e in msp.query("INSERT"):
    inserts.append({
        "block": e.dxf.name,
        "x": e.dxf.insert.x,
        "y": e.dxf.insert.y,
        "layer": e.dxf.layer,
        "rotation": float(getattr(e.dxf, "rotation", 0.0)),
        "scale_x": float(getattr(e.dxf, "xscale", 1.0)),
        "scale_y": float(getattr(e.dxf, "yscale", 1.0)),
    })
df_ins = pd.DataFrame(inserts)

# 1C) Lines and polylines (pipes)
pipes = []
for e in msp.query("LINE LWPOLYLINE"):
    if e.dxftype() == "LINE":
        p1 = (e.dxf.start.x, e.dxf.start.y)
        p2 = (e.dxf.end.x, e.dxf.end.y)
        pts = [p1, p2]
    else:
        # LWPOLYLINE points
        pts = [(p[0], p[1]) for p in e.get_points("xy")]
    pipes.append({"type": e.dxftype(), "layer": e.dxf.layer, "pts": pts})

df_pipe = pd.DataFrame(pipes)

print("TEXT:", len(df_text), "INSERT:", len(df_ins), "PIPES:", len(df_pipe))
df_ins.head()


In [ ]:
from shapely.geometry import LineString, Point
import networkx as nx

# Convert pipes to shapely lines
pipe_lines = []
for i,r in df_pipe.iterrows():
    if len(r["pts"]) >= 2:
        pipe_lines.append((i, r["layer"], LineString(r["pts"])))

# Create graph nodes from inserts
G = nx.Graph()
for i,r in df_ins.iterrows():
    G.add_node(
        f"ins_{i}",
        kind="insert",
        block=r["block"],
        layer=r["layer"],
        x=float(r["x"]),
        y=float(r["y"])
    )

# Helper: distance from node to pipe
def node_to_pipe_dist(node_id, pipe_geom):
    x, y = G.nodes[node_id]["x"], G.nodes[node_id]["y"]
    return pipe_geom.distance(Point(x, y))

# Connect nodes that lie close to the same pipe segment
DIST_THRESH = 5.0  # you may tune depending on drawing units
for pid, layer, geom in pipe_lines:
    near = []
    for n in G.nodes:
        d = node_to_pipe_dist(n, geom)
        if d <= DIST_THRESH:
            near.append((n, d))
    near = sorted(near, key=lambda t: t[1])

    # connect near nodes along that pipe (pairwise)
    for a in range(len(near)-1):
        u, du = near[a]
        v, dv = near[a+1]
        if u != v:
            G.add_edge(u, v, edge_type="process_pipe", pipe_id=int(pid), pipe_layer=layer)

print("Graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))
# plot pipes
for _,_,geom in pipe_lines[:500]:  # cap if huge
    xs, ys = geom.xy
    plt.plot(xs, ys, linewidth=0.5)

# plot nodes
xs = [G.nodes[n]["x"] for n in G.nodes]
ys = [G.nodes[n]["y"] for n in G.nodes]
plt.scatter(xs, ys, s=8)

plt.title("DXF P&ID: Pipes + Insert Nodes (sanity check)")
plt.axis("equal")
plt.show()


In [ ]:
xs = df_ins["x"].values
ys = df_ins["y"].values

print("X range:", xs.min(), xs.max())
print("Y range:", ys.min(), ys.max())


In [ ]:
import numpy as np

# Robust center
mx, my = np.median(xs), np.median(ys)

# Distance from center
dist = np.sqrt((xs-mx)**2 + (ys-my)**2)

# Keep only nodes within reasonable radius
KEEP_RADIUS = np.percentile(dist, 95)  # keeps 95% of symbols
mask = dist <= KEEP_RADIUS

df_ins_clean = df_ins[mask].copy()
print("Before:", len(df_ins), "After:", len(df_ins_clean))


In [ ]:
# Normalize to zero mean, unit scale
x_mean, y_mean = df_ins_clean["x"].mean(), df_ins_clean["y"].mean()
x_std,  y_std  = df_ins_clean["x"].std(),  df_ins_clean["y"].std()

df_ins_clean["x_norm"] = (df_ins_clean["x"] - x_mean) / x_std
df_ins_clean["y_norm"] = (df_ins_clean["y"] - y_mean) / y_std


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(df_ins_clean["x_norm"], df_ins_clean["y_norm"], s=10)
plt.title("Normalized P&ID Insert Nodes (Sanity Check)")
plt.axis("equal")
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np

coords = df_ins_clean[["x_norm", "y_norm"]].values

# DBSCAN groups spatial clusters
clustering = DBSCAN(eps=0.6, min_samples=5).fit(coords)
df_ins_clean["cluster"] = clustering.labels_

df_ins_clean["cluster"].value_counts()


In [ ]:
main_cluster = df_ins_clean["cluster"].value_counts().idxmax()
df_main = df_ins_clean[df_ins_clean["cluster"] == main_cluster].copy()

print("Keeping cluster:", main_cluster, "Nodes:", len(df_main))


In [ ]:
x_mean = df_main["x"].mean()
y_mean = df_main["y"].mean()
x_std  = df_main["x"].std()
y_std  = df_main["y"].std()

df_main["x_n"] = (df_main["x"] - x_mean) / x_std
df_main["y_n"] = (df_main["y"] - y_mean) / y_std


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(df_main["x_n"], df_main["y_n"], s=15)
plt.title("Final ML-Ready P&ID Geometry (Single Cluster)")
plt.axis("equal")
plt.show()


In [ ]:
import networkx as nx
import numpy as np

G = nx.Graph()

for i,r in df_main.iterrows():
    G.add_node(
        f"ins_{i}",
        block=str(r["block"]),
        layer=str(r["layer"]),
        x=float(r["x_n"]),
        y=float(r["y_n"])
    )

print("Nodes:", G.number_of_nodes())


In [ ]:
from sklearn.neighbors import NearestNeighbors

node_ids = list(G.nodes())
coords = np.array([[G.nodes[n]["x"], G.nodes[n]["y"]] for n in node_ids], dtype=float)

K = 6  # nearest neighbors per node (tune 4–10)
nbrs = NearestNeighbors(n_neighbors=K+1, algorithm="ball_tree").fit(coords)
distances, indices = nbrs.kneighbors(coords)

# add edges excluding self
for i in range(len(node_ids)):
    for jpos in range(1, K+1):
        j = indices[i, jpos]
        d = distances[i, jpos]
        if d < 0.8:  # distance cutoff in normalized coords (tune)
            u, v = node_ids[i], node_ids[j]
            if u != v:
                G.add_edge(u, v, edge_type="knn_candidate", dist=float(d))

print("Edges:", G.number_of_edges())


In [ ]:
import pandas as pd
import json

nodes_df = pd.DataFrame([
    {"id": n, **G.nodes[n]} for n in G.nodes()
])

edges_df = pd.DataFrame([
    {"u": u, "v": v, **attr} for u,v,attr in G.edges(data=True)
])

nodes_df.to_csv("nodes.csv", index=False)
edges_df.to_csv("edges.csv", index=False)

pid_json = {"nodes": nodes_df.to_dict("records"), "edges": edges_df.to_dict("records")}
with open("pid.json","w") as f:
    json.dump(pid_json, f, indent=2)

print("Saved nodes.csv, edges.csv, pid.json")


In [ ]:
deg = np.array([d for _,d in G.degree()])
print("Median degree:", np.median(deg))
print("Max degree:", deg.max())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,7))
for u,v,attr in list(G.edges(data=True))[:800]:
    plt.plot([G.nodes[u]["x"], G.nodes[v]["x"]],
             [G.nodes[u]["y"], G.nodes[v]["y"]],
             linewidth=0.4)

plt.scatter(coords[:,0], coords[:,1], s=10)
plt.title("Graph check: edges are local (KNN)")
plt.axis("equal")
plt.show()


In [ ]:
import numpy as np

def find_tag_xy(df_text, tag):
    pat = rf"\b{tag.replace('-','[-\s]?')}\b"
    hits = df_text[df_text["text"].astype(str).str.contains(pat, case=False, regex=True)]
    if len(hits) == 0:
        raise ValueError(f"Tag not found: {tag}")
    return hits.iloc[0][["x","y"]].values.astype(float)

def nearest_insert_index(df_main, xy):
    pts = df_main[["x","y"]].values.astype(float)
    d = np.sqrt(((pts - xy)**2).sum(axis=1))
    return df_main.iloc[np.argmin(d)].name

reactor_tags = ["R-100","R-200","R-300"]
reactor_nodes = {}

for t in reactor_tags:
    xy = find_tag_xy(df_text, t)
    ins_idx = nearest_insert_index(df_main, xy)
    reactor_nodes[t] = f"ins_{ins_idx}"
    # stamp role
    G.nodes[reactor_nodes[t]]["unit_tag"] = t
    G.nodes[reactor_nodes[t]]["unit_type"] = "reactor"

G.nodes[reactor_nodes["R-300"]]["reactor_stage"] = "catalytic_fixed_bed"
G.nodes[reactor_nodes["R-100"]]["reactor_stage"] = "acidification"
G.nodes[reactor_nodes["R-200"]]["reactor_stage"] = "acidification"

reactor_nodes


In [ ]:
import torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.nn import SAGEConv

# --- index mapping
node_list = list(G.nodes())
idx = {n:i for i,n in enumerate(node_list)}

r100 = idx[reactor_nodes["R-100"]]
r200 = idx[reactor_nodes["R-200"]]
r300 = idx[reactor_nodes["R-300"]]

def block_hash(s, k=16):
    h = abs(hash(s)) % (2**31)
    v = np.zeros(k, dtype=np.float32)
    v[h % k] = 1.0
    return v

# base node features: (x,y) + hashed block name
base = []
for n in node_list:
    base.append(np.concatenate([
        np.array([G.nodes[n]["x"], G.nodes[n]["y"]], np.float32),
        block_hash(G.nodes[n].get("block","UNK"), 16)
    ]))
base = torch.tensor(np.vstack(base), dtype=torch.float32)

edges = [[idx[u], idx[v]] for u,v in G.edges()]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t().contiguous())
data_static = Data(x=base, edge_index=edge_index)

# Dynamic inputs (example):
# R100: [T,P,acid_dose,tau]
# R200: [T,P,acid_dose,tau]
# R300: [T,P,WHSV,H2_feed,catalyst_family_id,activity]
# catalyst_family_id: encode as 0..N (NiMo,CoMo,Ni2P,Ru,...)
DYN_DIM = 4 + 4 + 6

def make_graph_sample(x_dyn):
    dyn = torch.zeros((data_static.num_nodes, DYN_DIM), dtype=torch.float32)
    dyn[r100, 0:4] = torch.tensor(x_dyn[0:4])
    dyn[r200, 4:8] = torch.tensor(x_dyn[4:8])
    dyn[r300, 8:14] = torch.tensor(x_dyn[8:14])
    return Data(x=torch.cat([data_static.x, dyn], dim=1), edge_index=data_static.edge_index)

class BioOilKPI_GNN(nn.Module):
    def __init__(self, in_dim, hid=96, out_dim=5):
        super().__init__()
        self.c1 = SAGEConv(in_dim, hid)
        self.c2 = SAGEConv(hid, hid)
        self.c3 = SAGEConv(hid, hid)
        self.head = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, out_dim))
    def forward(self, g):
        h = F.relu(self.c1(g.x, g.edge_index))
        h = F.relu(self.c2(h, g.edge_index))
        h = F.relu(self.c3(h, g.edge_index))
        hg = h.mean(dim=0)               # plant-level KPI readout
        return self.head(hg)

model = BioOilKPI_GNN(in_dim=data_static.x.shape[1]+DYN_DIM, hid=96, out_dim=5)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# -----------------------------
# 1) Fixed 5-node "P&ID" graph
# -----------------------------
N_NODES = 5
edges = [(0,1),(1,2),(2,3),(3,4)]
A = np.zeros((N_NODES,N_NODES), dtype=np.float32)
for u,v in edges:
    A[u,v] = 1.0; A[v,u] = 1.0
A += np.eye(N_NODES, dtype=np.float32)  # self loops
An = A / (A.sum(axis=1, keepdims=True) + 1e-6)        # row-normalized adjacency
An_t = torch.tensor(An, dtype=torch.float32)

# node type one-hot: Feed, Acid, Acid, 13X-bed, Product
node_types = np.array([0,1,1,2,3], dtype=np.int64)
num_types = 4
type_onehot = np.eye(num_types, dtype=np.float32)[node_types]  # (5,4)
BASE = type_onehot.shape[1]

# -----------------------------------------
# 2) Toy "truth" function (13X dries water)
# -----------------------------------------
def clamp(x, lo, hi): return np.clip(x, lo, hi)

def truth(u):
    """
    u = [T1,P1,acid1,tau1,  T2,P2,acid2,tau2,  T3,P3,WHSV,H2, zfrac, mload, act,  Fkgph,O0,W0,V0]
    outputs = [Yield, Moisture, Oxygenates, Viscosity, Stability, Q_kW, dP_bar]
    """
    (T1,P1,a1,tau1, T2,P2,a2,tau2, T3,P3,WHSV,H2, zfrac,mload,act, Fkgph,O0,W0,V0) = u

    sev1 = (a1*tau1)*(T1/480.0)**0.6
    sev2 = (a2*tau2)*(T2/500.0)**0.6

    Oa = O0*(1-0.10*clamp(sev1,0,1.5))*(1-0.12*clamp(sev2,0,1.5))
    Wa = W0*(1-0.06*clamp(sev1,0,1.5))*(1-0.06*clamp(sev2,0,1.5))
    Va = V0*(1-0.04*clamp(sev1,0,1.5))*(1-0.05*clamp(sev2,0,1.5))
    Ya = 0.62*(1-0.01*clamp(sev1,0,2))*(1-0.01*clamp(sev2,0,2))

    sev3 = (T3/650.0)**1.2*(P3/30.0)**0.5*(H2**0.6)*act/(clamp(WHSV,0.2,10.0))

    # 13X effect: strong water adsorption (higher at lower T); modest oxygenate trapping
    ads_water = zfrac*1.2*(650.0/max(T3,1e-3))**0.7
    ads_oxyg  = zfrac*0.25

    # "modified 13X" proxy improves deoxygenation slightly (metal loading)
    mod_boost = 1 + 0.15*clamp(mload,0,1.0)

    Oout = Oa*(1-0.45*clamp(sev3*mod_boost,0,1.5))*(1-ads_oxyg)
    Wout = Wa*(1-0.18*clamp(sev3,0,1.5))*(1-0.55*clamp(ads_water,0,0.8))
    Vout = Va*(1-0.30*clamp(sev3,0,1.5))*(1-0.10*clamp(ads_oxyg,0,0.3))

    over = max(0.0, sev3-1.0)           # over-severity -> cracking -> yield loss
    Yout = Ya*(1-0.12*over)

    stability = 0.3 + 1.2*(O0-Oout) + 0.8*(W0-Wout)
    stability = clamp(stability, 0.0, 2.5)

    Q  = (0.8*sev3*Fkgph*10) + (0.02*(T3-300)*Fkgph)
    dP = 0.02*Fkgph*(P3/30.0) + 0.01*Fkgph*(1/max(WHSV,0.2))

    return np.array([Yout,Wout,Oout,Vout,stability,Q,dP], dtype=np.float32)

# -----------------------------
# 3) Generate dataset (POC scale)
# -----------------------------
rng = np.random.default_rng(7)
N = 2000
X = np.zeros((N,19), dtype=np.float32)
Y = np.zeros((N,7), dtype=np.float32)

for i in range(N):
    T1=rng.uniform(430,520); P1=rng.uniform(5,20);  a1=rng.uniform(0.1,1.0); tau1=rng.uniform(0.5,3.0)
    T2=rng.uniform(440,540); P2=rng.uniform(5,25);  a2=rng.uniform(0.1,1.0); tau2=rng.uniform(0.5,3.0)
    T3=rng.uniform(560,720); P3=rng.uniform(10,60); WHSV=rng.uniform(0.5,5.0); H2=rng.uniform(0.5,5.0)
    zfrac=rng.uniform(0.2,1.0); mload=rng.uniform(0.0,1.0); act=rng.uniform(0.6,1.0)
    Fkgph=rng.uniform(0.2,1.0)                  # around 10 kg/day ≈ 0.417 kg/h
    O0=rng.uniform(0.25,0.45); W0=rng.uniform(0.15,0.35); V0=rng.uniform(50,250)

    u=np.array([T1,P1,a1,tau1,T2,P2,a2,tau2,T3,P3,WHSV,H2,zfrac,mload,act,Fkgph,O0,W0,V0],dtype=np.float32)
    X[i]=u; Y[i]=truth(u)

# normalize inputs
X_mean, X_std = X.mean(axis=0), X.std(axis=0)+1e-6
Xn = (X - X_mean)/X_std

# build node features per sample (5 nodes × (type + dynamic))
DYN = 19
Fdim = BASE + DYN

def build_node_features(xrow):
    dyn = np.zeros((N_NODES,DYN), dtype=np.float32)
    dyn[0,-4:] = xrow[-4:]      # Feed: [Fkgph, O0, W0, V0]
    dyn[1,0:4] = xrow[0:4]      # R-100
    dyn[2,4:8] = xrow[4:8]      # R-200
    dyn[3,8:15]= xrow[8:15]     # R-300 (13X bed inputs)
    return np.concatenate([type_onehot, dyn], axis=1)

X_nodes = np.stack([build_node_features(Xn[i]) for i in range(N)], axis=0)  # (N,5,Fdim)

# train/test split
tr, te = train_test_split(np.arange(N), test_size=0.2, random_state=0)
Xtr = torch.tensor(X_nodes[tr], dtype=torch.float32)
Ytr = torch.tensor(Y[tr], dtype=torch.float32)
Xte = torch.tensor(X_nodes[te], dtype=torch.float32)
Yte = torch.tensor(Y[te], dtype=torch.float32)

# -----------------------------
# 4) Message-passing GNN model
# -----------------------------
class SimpleGNN(nn.Module):
    def __init__(self, in_dim, hid=96, out_dim=7):
        super().__init__()
        self.l1 = nn.Linear(in_dim, hid)
        self.l2 = nn.Linear(hid, hid)
        self.l3 = nn.Linear(hid, hid)
        self.out= nn.Linear(hid, out_dim)

    def forward(self, x_nodes, A_norm):
        # x_nodes: (B, N, F)
        h = F.relu(self.l1(x_nodes))
        h = torch.matmul(A_norm, h)      # message passing 1
        h = F.relu(self.l2(h))
        h = torch.matmul(A_norm, h)      # message passing 2
        h = F.relu(self.l3(h))
        hg = h.mean(dim=1)               # graph pooling
        return self.out(hg)

model = SimpleGNN(Fdim, hid=96, out_dim=7)
opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-4)

# training
batch = 128
for epoch in range(1, 26):
    model.train()
    order = torch.randperm(Xtr.shape[0])
    for i in range(0, Xtr.shape[0], batch):
        sel = order[i:i+batch]
        pred = model(Xtr[sel], An_t)
        loss = F.mse_loss(pred, Ytr[sel])
        opt.zero_grad(); loss.backward(); opt.step()

    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            P = model(Xte, An_t)
        rmse = torch.sqrt(torch.mean((P - Yte)**2, dim=0)).numpy()
        print(f"epoch {epoch:02d} test_RMSE:", np.round(rmse, 4))

# report KPIs
names = ["Yield","Moisture","Oxygenates","Viscosity","Stability","Q_kW","dP_bar"]
model.eval()
with torch.no_grad():
    P = model(Xte, An_t).numpy()
Yte_np = Yte.numpy()

for j,nm in enumerate(names):
    print(nm, "R2=", round(r2_score(Yte_np[:,j], P[:,j]), 4),
              "MAE=", round(mean_absolute_error(Yte_np[:,j], P[:,j]), 4))

# SCADA-style what-if: change only R-300 temperature (T3) and see outputs change
def predict_one(u_raw):
    x = (u_raw - X_mean)/X_std
    xnodes = torch.tensor(build_node_features(x)[None,:,:], dtype=torch.float32)
    with torch.no_grad():
        out = model(xnodes, An_t).numpy().ravel()
    return dict(zip(names, out))

u0 = X[0].copy()
base = predict_one(u0)
u1 = u0.copy(); u1[8] = 700.0   # T3 index in vector = 8
hot  = predict_one(u1)

print("\nBase:", base)
print("R-300 T up:", hot)


In [ ]:
import pandas as pd

# --- Table 6 (design variables) ---
design = pd.DataFrame({
    "Run": list(range(1, 16)),
    "SiO2_Al2O3": [50,80,80,80,50,80,23,50,23,50,23,23,50,50,50],
    "ZSM5_wt_pct": [22,30,22,13,30,22,22,22,13,30,30,22,13,13,22],
    "Temp_C": [500,500,450,500,450,550,550,500,500,550,500,450,550,450,500],
    "CTO": [5,6,5,4,6,5,5,5,4,6,6,5,4,4,5],
})

# --- Table 12 (measured responses) ---
resp = pd.DataFrame([
    [1,  73.73, 51.64, 4.14, 15.05, 23.28, 7.27, 14.82, 9.17],
    [2,  85.58, 64.78, 4.74, 19.21, 29.31, 8.34, 12.47, 11.52],
    [3,  71.53, 53.85, 2.42, 14.99, 25.30, 10.02, 7.66, 11.14],
    [4,  74.71, 56.30, 3.17, 15.20, 27.93, 10.63, 7.78, 10.00],
    [5,  59.08, 46.67, 2.93, 13.79, 21.73, 6.65, 5.77, 8.22],
    [6,  66.22, 53.87, 5.27, 18.00, 21.99, 6.33, 6.02, 8.61],
    [7,  70.84, 57.06, 6.57, 17.23, 23.70, 5.90, 7.87, 9.56],
    [8,  66.57, 40.77, 2.37, 12.05, 18.88, 7.63, 18.17, 7.47],
    [9,  63.59, 53.22, 5.67, 12.52, 23.47, 6.14, 4.23, 11.55],
    [10, 72.95, 52.09, 6.24, 12.48, 22.88, 6.66, 14.20, 10.50],
    [11, 72.02, 43.86, 5.47, 10.15, 19.04, 8.01, 20.14, 9.21],
    [12, 65.47, 43.52, 1.45, 11.33, 23.35, 10.27, 11.67, 7.39],
    [13, 73.08, 49.91, 5.36, 14.05, 21.83, 7.73, 15.44, 8.68],
    [14, 81.27, 50.21, 2.42, 12.84, 24.88, 12.54, 18.52, 10.08],
    [15, 70.50, 42.90, 2.77, 10.74, 21.80, 9.06, 18.54, 7.58],
], columns=[
    "Run","MassBalance_pct","Conversion_pct","DryGas_pct","LPG_pct",
    "GasolineNaphtha_pct","LCO_pct","BottomFrac_pct","Coke_pct"
])

df = design.merge(resp, on="Run")
df.to_csv("fcc_biooil_zeolite_dataset.csv", index=False)
print(df.head())


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["SiO2_Al2O3","ZSM5_wt_pct","Temp_C","CTO"]].values.astype(np.float32)
Y = df[["Conversion_pct","GasolineNaphtha_pct","Coke_pct","DryGas_pct","BottomFrac_pct"]].values.astype(np.float32)

Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.25, random_state=7)

sx = StandardScaler().fit(Xtr)
sy = StandardScaler().fit(Ytr)

Xtr_s = sx.transform(Xtr); Xte_s = sx.transform(Xte)
Ytr_s = sy.transform(Ytr); Yte_s = sy.transform(Yte)

Xtr_t = torch.tensor(Xtr_s); Ytr_t = torch.tensor(Ytr_s)
Xte_t = torch.tensor(Xte_s); Yte_t = torch.tensor(Yte_s)

model = nn.Sequential(
    nn.Linear(4, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, Y.shape[1])
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(1500):
    model.train()
    pred = model(Xtr_t)
    loss = loss_fn(pred, Ytr_t)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    te_pred = sy.inverse_transform(model(Xte_t).numpy())
print("Sample predictions (test):")
print(te_pred[:3])


In [ ]:
!pip -q install pymoo

import numpy as np
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize

def surrogate_predict(x_row):
    # x_row: [SiO2/Al2O3, ZSM5_wt%, Temp_C, CTO]
    x = np.array(x_row, dtype=np.float32).reshape(1, -1)
    x_s = sx.transform(x)
    with torch.no_grad():
        y_s = model(torch.tensor(x_s)).numpy()
    y = sy.inverse_transform(y_s)[0]
    # outputs: [Conversion, Gasoline, Coke, DryGas, BottomFrac]
    return y

class FCCOpt(Problem):
    def __init__(self):
        super().__init__(
            n_var=4, n_obj=4, n_constr=0,
            xl=np.array([23, 13, 450, 4]),
            xu=np.array([80, 30, 550, 6])
        )
    def _evaluate(self, X, out, *args, **kwargs):
        F = []
        for row in X:
            conv, gaso, coke, drygas, bottom = surrogate_predict(row)
            # minimize objectives (so negate gasoline for maximize)
            F.append([
                -gaso,
                coke,
                drygas,
                bottom
            ])
        out["F"] = np.array(F)

problem = FCCOpt()
algo = NSGA2(pop_size=80, sampling=FloatRandomSampling())

res = minimize(problem, algo, ('n_gen', 120), verbose=False)
pareto = res.X

print("Top 5 Pareto candidates:")
for i in range(5):
    print(pareto[i], "=>", surrogate_predict(pareto[i]))


In [ ]:
import pandas as pd

# --- Table 6 (design variables) ---
design = pd.DataFrame({
    "Run": list(range(1, 16)),
    "SiO2_Al2O3": [50,80,80,80,50,80,23,50,23,50,23,23,50,50,50],
    "ZSM5_wt_pct": [22,30,22,13,30,22,22,22,13,30,30,22,13,13,22],
    "Temp_C": [500,500,450,500,450,550,550,500,500,550,500,450,550,450,500],
    "CTO": [5,6,5,4,6,5,5,5,4,6,6,5,4,4,5],
})

# --- Table 12 (measured responses) ---
resp = pd.DataFrame([
    [1,  73.73, 51.64, 4.14, 15.05, 23.28, 7.27, 14.82, 9.17],
    [2,  85.58, 64.78, 4.74, 19.21, 29.31, 8.34, 12.47, 11.52],
    [3,  71.53, 53.85, 2.42, 14.99, 25.30, 10.02, 7.66, 11.14],
    [4,  74.71, 56.30, 3.17, 15.20, 27.93, 10.63, 7.78, 10.00],
    [5,  59.08, 46.67, 2.93, 13.79, 21.73, 6.65, 5.77, 8.22],
    [6,  66.22, 53.87, 5.27, 18.00, 21.99, 6.33, 6.02, 8.61],
    [7,  70.84, 57.06, 6.57, 17.23, 23.70, 5.90, 7.87, 9.56],
    [8,  66.57, 40.77, 2.37, 12.05, 18.88, 7.63, 18.17, 7.47],
    [9,  63.59, 53.22, 5.67, 12.52, 23.47, 6.14, 4.23, 11.55],
    [10, 72.95, 52.09, 6.24, 12.48, 22.88, 6.66, 14.20, 10.50],
    [11, 72.02, 43.86, 5.47, 10.15, 19.04, 8.01, 20.14, 9.21],
    [12, 65.47, 43.52, 1.45, 11.33, 23.35, 10.27, 11.67, 7.39],
    [13, 73.08, 49.91, 5.36, 14.05, 21.83, 7.73, 15.44, 8.68],
    [14, 81.27, 50.21, 2.42, 12.84, 24.88, 12.54, 18.52, 10.08],
    [15, 70.50, 42.90, 2.77, 10.74, 21.80, 9.06, 18.54, 7.58],
], columns=[
    "Run","MassBalance_pct","Conversion_pct","DryGas_pct","LPG_pct",
    "GasolineNaphtha_pct","LCO_pct","BottomFrac_pct","Coke_pct"
])

df = design.merge(resp, on="Run")
df.to_csv("fcc_biooil_zeolite_dataset.csv", index=False)
print(df.head())


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["SiO2_Al2O3","ZSM5_wt_pct","Temp_C","CTO"]].values.astype(np.float32)
Y = df[["Conversion_pct","GasolineNaphtha_pct","Coke_pct","DryGas_pct","BottomFrac_pct"]].values.astype(np.float32)

Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.25, random_state=7)

sx = StandardScaler().fit(Xtr)
sy = StandardScaler().fit(Ytr)

Xtr_s = sx.transform(Xtr); Xte_s = sx.transform(Xte)
Ytr_s = sy.transform(Ytr); Yte_s = sy.transform(Yte)

Xtr_t = torch.tensor(Xtr_s); Ytr_t = torch.tensor(Ytr_s)
Xte_t = torch.tensor(Xte_s); Yte_t = torch.tensor(Yte_s)

model = nn.Sequential(
    nn.Linear(4, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, Y.shape[1])
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(1500):
    model.train()
    pred = model(Xtr_t)
    loss = loss_fn(pred, Ytr_t)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    te_pred = sy.inverse_transform(model(Xte_t).numpy())
print("Sample predictions (test):")
print(te_pred[:3])


In [ ]:
!pip -q install pymoo

import numpy as np
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize

def surrogate_predict(x_row):
    # x_row: [SiO2/Al2O3, ZSM5_wt%, Temp_C, CTO]
    x = np.array(x_row, dtype=np.float32).reshape(1, -1)
    x_s = sx.transform(x)
    with torch.no_grad():
        y_s = model(torch.tensor(x_s)).numpy()
    y = sy.inverse_transform(y_s)[0]
    # outputs: [Conversion, Gasoline, Coke, DryGas, BottomFrac]
    return y

class FCCOpt(Problem):
    def __init__(self):
        super().__init__(
            n_var=4, n_obj=4, n_constr=0,
            xl=np.array([23, 13, 450, 4]),
            xu=np.array([80, 30, 550, 6])
        )
    def _evaluate(self, X, out, *args, **kwargs):
        F = []
        for row in X:
            conv, gaso, coke, drygas, bottom = surrogate_predict(row)
            # minimize objectives (so negate gasoline for maximize)
            F.append([
                -gaso,
                coke,
                drygas,
                bottom
            ])
        out["F"] = np.array(F)

problem = FCCOpt()
algo = NSGA2(pop_size=80, sampling=FloatRandomSampling())

res = minimize(problem, algo, ('n_gen', 120), verbose=False)
pareto = res.X

print("Top 5 Pareto candidates:")
for i in range(5):
    print(pareto[i], "=>", surrogate_predict(pareto[i]))


In [ ]:
# !pip -q install ezdxf shapely networkx torch torch-geometric pymoo pandas numpy scikit-learn

import ezdxf, numpy as np, pandas as pd, networkx as nx
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree

dxf_path = "/content/CR11210004_P&ID OF IN SITU VAPOUR UPGRADATION.dxf"
doc = ezdxf.readfile(dxf_path)
msp = doc.modelspace()

# 1) Collect INSERT blocks (symbols/equipment/instruments)
insert_rows = []
for e in msp.query("INSERT"):
    p = e.dxf.insert
    insert_rows.append({
        "id": f"ins_{len(insert_rows)}",
        "block": e.dxf.name,
        "x": float(p.x), "y": float(p.y),
        "layer": e.dxf.layer
    })
ins = pd.DataFrame(insert_rows)

# 2) Collect LINE/LWPOLYLINE as pipes (geometry)
pipe_lines = []
for e in msp:
    if e.dxftype() == "LINE":
        p1, p2 = e.dxf.start, e.dxf.end
        pipe_lines.append(LineString([(p1.x,p1.y),(p2.x,p2.y)]))
    elif e.dxftype() == "LWPOLYLINE":
        pts = [(p[0], p[1]) for p in e.get_points()]
        if len(pts) >= 2:
            pipe_lines.append(LineString(pts))

# 3) Normalize + remove outliers (you already did similar)
XY = ins[["x","y"]].to_numpy()
med = np.median(XY, axis=0)
dist = np.linalg.norm(XY - med, axis=1)
keep = dist < np.quantile(dist, 0.98)    # drop far outliers
ins = ins.loc[keep].reset_index(drop=True)

XY = ins[["x","y"]].to_numpy()
mu, sig = XY.mean(axis=0), XY.std(axis=0) + 1e-9
ins["xn"] = (ins["x"]-mu[0])/sig[0]
ins["yn"] = (ins["y"]-mu[1])/sig[1]


In [ ]:
tol = 5.0  # CAD units tolerance; tune
points = [Point(xy) for xy in ins[["x","y"]].to_numpy()]
tree = STRtree(points)

G = nx.Graph()
for i, r in ins.iterrows():
    G.add_node(r["id"], block=r["block"], layer=r["layer"], x=r["xn"], y=r["yn"])

# pipe-based connections: if two inserts both lie near same pipe segment, connect
for line in pipe_lines:
    # buffer the line to catch nearby symbols
    buf = line.buffer(tol)
    idx = [j for j,p in enumerate(points) if buf.contains(p)]
    if len(idx) >= 2:
        for a in idx:
            for b in idx:
                if a < b:
                    G.add_edge(ins.loc[a,"id"], ins.loc[b,"id"], etype="pipe")


In [ ]:
reactor_tags = {
    "R-100": {"role":"acidification_1"},
    "R-200": {"role":"acidification_2"},
    "R-300": {"role":"acidification_3"},
    "R-101": {"role":"catalytic_fixed_bed"}
}

# Later: map these tags to node IDs (by nearest TEXT entity or by block name)
# For demo: you manually choose node ids once and store them.


In [ ]:
# Factors: acidity (SiO2/Al2O3), Temperature, CTO
# Responses: conversion, dry_gas, lpg, gasoline, lco, bottoms, coke, etc.

data = [
 # run, acidity, T, CTO, massbal, conv, drygas, lpg, gasoline, lco, bottoms, coke
 [1,  50, 500, 5, 73.73, 51.64, 4.14, 15.05, 23.28, 7.27, 14.82, 9.17],
 [2,  80, 500, 6, 85.58, 64.78, 4.74, 19.21, 29.31, 8.34, 12.47, 11.52],
 # ...continue filling from Table 12...
]
df = pd.DataFrame(data, columns=[
 "run","acidity","T_C","CTO","mass_balance","conversion",
 "dry_gas","LPG","gasoline_naphtha","LCO","bottoms","coke"
])


In [ ]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np

# Suppose you trained a surrogate model f(x)->y (even a RandomForest for demo)
# x = [acidity, T_C, CTO]

class FCCProblem(ElementwiseProblem):
    def __init__(self, model):
        super().__init__(n_var=3, n_obj=3, n_constr=0,
                         xl=np.array([23, 450, 4]),
                         xu=np.array([80, 550, 6]))
        self.model = model

    def _evaluate(self, x, out, *args, **kwargs):
        y = self.model.predict(x.reshape(1,-1))[0]  # returns [gasoline, coke, drygas] for demo
        gasoline, coke, drygas = y
        out["F"] = np.array([-gasoline, coke, drygas])  # maximize gasoline => minimize -gasoline

# algorithm = NSGA2(pop_size=80)
# res = minimize(FCCProblem(model), algorithm, ("n_gen", 120), seed=1, verbose=True)
# res.F gives Pareto front, res.X gives settings


In [ ]:
import pandas as pd

data = [
    # catalyst, metal_wt, SA, PV, PD, SiAl, T, cat_load, time, P,
    # O_wt, H2O_wt, HHV, visc, TAN, DOD
    ["Fe2O3_HZ", 1.27, 12.9, 0.070, 3.13, 6.94, 250, 2.5, 2.0, 1,
     53.8, 27.0, 17.7, 2.31, 139, 73.4],

    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 250, 2.5, 2.0, 1,
     40.5, 10.1, 20.9, 2.52, 166, 85.2],

    ["CuO_HZ", 2.32, 30.7, 0.086, 3.10, 5.69, 250, 2.5, 2.0, 1,
     40.2, 8.3, 20.9, 3.13, 118, 85.4],

    # Temperature sweep (ZnO/H-Z)
    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 275, 2.5, 2.0, 1,
     36.2, 8.9, 22.1, 2.35, 150, 88.1],

    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 300, 2.5, 2.0, 1,
     32.8, 7.5, 21.0, 2.10, 160, 90.2],

    # Catalyst loading sweep (ZnO/H-Z)
    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 275, 5.0, 2.0, 1,
     34.5, 7.9, 22.6, 2.20, 142, 89.5],

    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 275, 7.5, 2.0, 1,
     31.0, 6.8, 21.8, 2.05, 155, 91.0],
]

cols = [
    "Catalyst","Metal_wt","SurfaceArea","PoreVolume","PoreDiameter",
    "SiAl","Temp_C","CatLoading_wt","Time_h","Pressure_atm",
    "Oxygen_wt","Water_wt","HHV_MJkg","Viscosity_cP","TAN","DOD_pct"
]

df = pd.DataFrame(data, columns=cols)
df.to_csv("HDO_real_dataset.csv", index=False)
df


R-100 (acid) → R-200 (acid) → R-300 (HDO catalyst)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.preprocessing import StandardScaler

# Encode catalyst as integer
cat_map = {c:i for i,c in enumerate(df["Catalyst"].unique())}
df["CatID"] = df["Catalyst"].map(cat_map)

X = df[[
    "CatID","Metal_wt","SurfaceArea","PoreVolume","PoreDiameter",
    "SiAl","Temp_C","CatLoading_wt"
]].values

Y = df[[
    "Oxygen_wt","Water_wt","HHV_MJkg","Viscosity_cP","DOD_pct"
]].values

sx, sy = StandardScaler(), StandardScaler()
Xn, Yn = sx.fit_transform(X), sy.fit_transform(Y)

Xn = torch.tensor(Xn, dtype=torch.float32)
Yn = torch.tensor(Yn, dtype=torch.float32)

class HDO_GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(Xn.shape[1], 64)
        self.fc2 = nn.Linear(64, 64)
        self.out = nn.Linear(64, Yn.shape[1])

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

model = HDO_GNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(2000):
    pred = model(Xn)
    loss = F.mse_loss(pred, Yn)
    opt.zero_grad(); loss.backward(); opt.step()

print("Training complete")


In [ ]:
def predict_kpi(catalyst, T, loading):
    row = df[df["Catalyst"]==catalyst].iloc[0]
    x = np.array([[
        cat_map[catalyst],
        row.Metal_wt, row.SurfaceArea, row.PoreVolume,
        row.PoreDiameter, row.SiAl, T, loading
    ]])
    x = torch.tensor(sx.transform(x), dtype=torch.float32)
    y = sy.inverse_transform(model(x).detach().numpy())[0]
    return {
        "Oxygen_wt": round(y[0],2),
        "Water_wt": round(y[1],2),
        "HHV": round(y[2],2),
        "Viscosity": round(y[3],2),
        "DOD": round(y[4],2)
    }

# DEMO
print("Base:", predict_kpi("ZnO_HZ", 250, 2.5))
print("Temp ↑:", predict_kpi("ZnO_HZ", 300, 2.5))
print("Catalyst loading ↑:", predict_kpi("ZnO_HZ", 275, 5.0))
print("Change catalyst:", predict_kpi("CuO_HZ", 275, 2.5))


In [ ]:
!pip -q install pymoo
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np

class HDO_Opt(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=3, n_obj=4,
                         xl=[250,2.5,0],
                         xu=[300,7.5,len(cat_map)-1])

    def _evaluate(self, x, out, *args, **kwargs):
        T, load, catid = x
        cat = list(cat_map.keys())[int(round(catid))]
        kpi = predict_kpi(cat, T, load)
        out["F"] = [
            kpi["Oxygen_wt"],
            kpi["Water_wt"],
            kpi["Viscosity"],
            -kpi["DOD"]
        ]

algo = NSGA2(pop_size=80)
res = minimize(HDO_Opt(), algo, ("n_gen", 100), verbose=False)

res.X[:5]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 140


In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = "HDO_ML_Dataset.csv"   # if you uploaded it
df = pd.read_csv(CSV_PATH)

print(df.columns)
display(df)


In [ ]:
df2 = df[df["Catalyst"]!="Raw"].copy()
x = np.arange(len(df2))

plt.figure(figsize=(8,4))
plt.bar(x-0.2, df2["Oxygen_wt"], width=0.2, label="Oxygen_wt")
plt.bar(x,     df2["Water_wt"],  width=0.2, label="Water_wt")
plt.bar(x+0.2, df2["DOD_pct"],   width=0.2, label="DOD_%")
plt.xticks(x, df2["Catalyst"], rotation=25, ha="right")
plt.title("HDO KPIs (from extracted Table 4)")
plt.grid(axis="y")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
!pip -q install torch

import torch
import torch.nn as nn
import torch.nn.functional as F

train_df = df[df["Catalyst"]!="Raw"].copy().reset_index(drop=True)

cats = train_df["Catalyst"].unique().tolist()
cat_to_id = {c:i for i,c in enumerate(cats)}
train_df["CatID"] = train_df["Catalyst"].map(cat_to_id)

# Inputs at R-300 (catalyst + setpoints)
X_num = train_df[["Si/Al","SurfaceArea_m2g","PoreVolume_ccg","PoreDiameter_nm","Temp_C","CatLoading_wt"]].values.astype(np.float32)
cat_ids = train_df["CatID"].values.astype(np.int64)

# Targets (KPIs)
Y_cols = ["Oxygen_wt","Water_wt","HHV_MJkg","Viscosity_cP","DOD_pct"]
Y = train_df[Y_cols].values.astype(np.float32)

# Normalize
X_mu, X_sig = X_num.mean(axis=0), X_num.std(axis=0) + 1e-6
Y_mu, Y_sig = Y.mean(axis=0), Y.std(axis=0) + 1e-6
Xn = (X_num - X_mu)/X_sig
Yn = (Y - Y_mu)/Y_sig

# Chain adjacency (4 nodes)
N=4
A = np.zeros((N,N), dtype=np.float32)
for u,v in [(0,1),(1,2),(2,3)]:
    A[u,v]=1; A[v,u]=1
A += np.eye(N, dtype=np.float32)
A = A / (A.sum(axis=1, keepdims=True)+1e-6)
A = torch.tensor(A, dtype=torch.float32)

def build_nodes(x_row):
    nodes = np.zeros((N,6), dtype=np.float32)
    nodes[2,:] = x_row  # only R-300 has features
    return nodes

X_nodes = torch.tensor(np.stack([build_nodes(Xn[i]) for i in range(len(Xn))]), dtype=torch.float32)
cat_ids_t = torch.tensor(cat_ids, dtype=torch.long)
Yn_t = torch.tensor(Yn, dtype=torch.float32)

class ChainGNN(nn.Module):
    def __init__(self, n_cat, emb=8, hid=64, out_dim=5):
        super().__init__()
        self.emb = nn.Embedding(n_cat, emb)
        self.lin_in = nn.Linear(6+emb, hid)
        self.lin_mp = nn.Linear(hid, hid)
        self.lin_out = nn.Linear(hid, out_dim)

    def forward(self, x_nodes, cat_id, A):
        B,N,_ = x_nodes.shape
        e = self.emb(cat_id)[:,None,:].repeat(1,N,1)
        h = torch.cat([x_nodes, e], dim=-1)
        h = F.relu(self.lin_in(h))
        h = torch.matmul(A, h)
        h = F.relu(self.lin_mp(h))
        h = torch.matmul(A, h)
        return self.lin_out(h[:,3,:])

model = ChainGNN(n_cat=len(cats), emb=8, hid=64, out_dim=len(Y_cols))
opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-4)

for epoch in range(1, 1201):
    pred = model(X_nodes, cat_ids_t, A)
    loss = F.mse_loss(pred, Yn_t)
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 300 == 0:
        print("epoch", epoch, "loss", float(loss))

print("trained")


In [ ]:
!pip -q install ipywidgets

import ipywidgets as widgets
from IPython.display import display

def predict_kpi(catalyst, T=250.0, load=2.5):
    base = train_df[train_df["Catalyst"]==catalyst].iloc[0]
    x_num = np.array([
        base["Si/Al"], base["SurfaceArea_m2g"], base["PoreVolume_ccg"], base["PoreDiameter_nm"],
        T, load
    ], dtype=np.float32)

    x_n = (x_num - X_mu)/X_sig
    nodes = np.zeros((1,4,6), dtype=np.float32)
    nodes[0,2,:] = x_n
    nodes_t = torch.tensor(nodes, dtype=torch.float32)
    cid = torch.tensor([cat_to_id[catalyst]], dtype=torch.long)

    with torch.no_grad():
        y_n = model(nodes_t, cid, A).numpy()[0]
    y = y_n * Y_sig + Y_mu
    return dict(zip(Y_cols, y.tolist()))

def live(catalyst, T, load):
    k = predict_kpi(catalyst, float(T), float(load))
    print({kk: round(v,3) for kk,v in k.items()})

    Ts = np.linspace(230, 320, 19)
    oxy = [predict_kpi(catalyst, float(t), float(load))["Oxygen_wt"] for t in Ts]
    dod = [predict_kpi(catalyst, float(t), float(load))["DOD_pct"] for t in Ts]

    plt.figure(figsize=(7,4))
    plt.plot(Ts, oxy, label="Oxygen_wt")
    plt.plot(Ts, dod, label="DOD_%")
    plt.title(f"{catalyst} | load={load} wt%")
    plt.xlabel("Temperature (°C)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

w_cat = widgets.Dropdown(options=cats, value=cats[0], description="Catalyst")
w_T   = widgets.FloatSlider(value=250, min=230, max=320, step=1, description="T (°C)")
w_L   = widgets.FloatSlider(value=2.5, min=2.5, max=10.0, step=0.5, description="Load")

ui = widgets.VBox([w_cat, w_T, w_L])
out = widgets.interactive_output(live, {"catalyst": w_cat, "T": w_T, "load": w_L})
display(ui, out)


In [ ]:
!pip -q install pymoo

import numpy as np
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

class HDO_MOO(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=3, n_obj=4, n_constr=0,
                         xl=np.array([230.0, 2.5, 0.0]),
                         xu=np.array([320.0, 10.0, float(len(cats)-1)]))

    def _evaluate(self, x, out, *args, **kwargs):
        T, load, cid = x
        cid = int(np.clip(np.round(cid), 0, len(cats)-1))
        cat = cats[cid]
        k = predict_kpi(cat, float(T), float(load))
        out["F"] = np.array([
            k["Oxygen_wt"],
            k["Water_wt"],
            k["Viscosity_cP"],
            -k["DOD_pct"]
        ], dtype=float)

res = minimize(HDO_MOO(), NSGA2(pop_size=80), ("n_gen", 120), seed=1, verbose=False)

print("First 10 Pareto samples:")
for i in range(min(10, len(res.X))):
    T, load, cid = res.X[i]
    cid = int(np.round(cid))
    cat = cats[cid]
    print(i, cat, "T=",round(T,1), "load=",round(load,2), {k: round(v,2) for k,v in predict_kpi(cat,T,load).items()})


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

F = res.F  # shape: (n_solutions, 4)
# F columns: [Oxygen_wt, Water_wt, Viscosity, -DOD]
oxygen = F[:,0]
water  = F[:,1]
visc   = F[:,2]
neg_dod= F[:,3]
dod    = -neg_dod

# Optional: pick a "best compromise" point by a simple normalized score
Fn = (F - F.min(axis=0)) / (F.ptp(axis=0) + 1e-9)
score = Fn[:,0] + Fn[:,1] + 0.2*Fn[:,2] + 0.2*Fn[:,3]   # tune weights
best_i = int(np.argmin(score))

def scatter2(x, y, xlabel, ylabel, title, best_idx=None):
    plt.figure(figsize=(6.5,4.5))
    plt.scatter(x, y, s=18)
    if best_idx is not None:
        plt.scatter([x[best_idx]], [y[best_idx]], s=120, marker="*", label="Best-compromise")
        plt.legend()
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

scatter2(oxygen, dod,  "Oxygen_wt (min)", "DOD_% (max)", "Pareto: Oxygen vs DOD", best_i)
scatter2(water,  dod,  "Water_wt (min)",  "DOD_% (max)", "Pareto: Water vs DOD", best_i)
scatter2(oxygen, visc, "Oxygen_wt (min)", "Viscosity (min)", "Pareto: Oxygen vs Viscosity", best_i)
scatter2(water,  visc, "Water_wt (min)",  "Viscosity (min)", "Pareto: Water vs Viscosity", best_i)

print("Best-compromise decision variables:", res.X[best_i])
print("Best-compromise objectives [O2, H2O, Visc, -DOD]:", res.F[best_i])


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 140


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

F = res.F  # shape: (n_solutions, 4)
# F columns: [Oxygen_wt, Water_wt, Viscosity, -DOD]
oxygen = F[:,0]
water  = F[:,1]
visc   = F[:,2]
neg_dod= F[:,3]
dod    = -neg_dod

# Optional: pick a "best compromise" point by a simple normalized score
Fn = (F - F.min(axis=0)) / (np.ptp(F, axis=0) + 1e-9)
score = Fn[:,0] + Fn[:,1] + 0.2*Fn[:,2] + 0.2*Fn[:,3]   # tune weights
best_i = int(np.argmin(score))

def scatter2(x, y, xlabel, ylabel, title, best_idx=None):
    plt.figure(figsize=(6.5,4.5))
    plt.scatter(x, y, s=18)
    if best_idx is not None:
        plt.scatter([x[best_idx]], [y[best_idx]], s=120, marker="*", label="Best-compromise")
        plt.legend()
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

scatter2(oxygen, dod,  "Oxygen_wt (min)", "DOD_% (max)", "Pareto: Oxygen vs DOD", best_i)
scatter2(water,  dod,  "Water_wt (min)",  "DOD_% (max)", "Pareto: Water vs DOD", best_i)
scatter2(oxygen, visc, "Oxygen_wt (min)", "Viscosity (min)", "Pareto: Oxygen vs Viscosity", best_i)
scatter2(water,  visc, "Water_wt (min)",  "Viscosity (min)", "Pareto: Water vs Viscosity", best_i)

print("Best-compromise decision variables:", res.X[best_i])
print("Best-compromise objectives [O2, H2O, Visc, -DOD]:", res.F[best_i])

In [ ]:
# Show top 10 by the same compromise score
top = np.argsort(score)[:10]
for j,i in enumerate(top):
    T, load, cid = res.X[i]
    cid = int(np.round(cid))
    cat = cats[cid]
    print(j, "|", cat, "| T=", round(T,1), "load=", round(load,2),
          "| O2=", round(oxygen[i],2),
          "H2O=", round(water[i],2),
          "Visc=", round(visc[i],2),
          "DOD=", round(dod[i],2))


In [ ]:
import pandas as pd

data = [
    ("Fe2O3/H-Z", 279.3, 6.67, 53.8, 27.0, 17.7, 2.31, 73.4),
    ("CuO/H-Z",   278.3, 8.21, 40.19, 8.29, 20.9, 3.13, 85.41),
    ("CuO/H-Z",   282.4, 9.46, 40.19, 8.29, 20.9, 3.13, 85.41),
    ("Fe2O3/H-Z", 306.5, 4.96, 53.8, 27.0, 17.7, 2.31, 73.4),
    ("ZnO/H-Z",   253.2, 5.8,  40.51,10.11,20.9, 2.52, 85.19),
    ("ZnO/H-Z",   250.3, 2.67, 40.51,10.11,20.9, 2.52, 85.19),
    ("ZnO/H-Z",   278.7, 9.4,  40.51,10.11,20.9, 2.52, 85.19),
    ("Fe2O3/H-Z", 312.6, 5.46, 53.8, 27.0, 17.7, 2.31, 73.4),
    ("Fe2O3/H-Z", 253.4, 4.57, 53.8, 27.0, 17.7, 2.31, 73.4),
    ("Fe2O3/H-Z", 244.1, 4.96, 53.8, 27.0, 17.7, 2.31, 73.4),
]
dfp = pd.DataFrame(data, columns=["Catalyst","T","Load","Oxygen","Water","HHV","Viscosity","DOD"])
dfp


In [ ]:
import numpy as np

labels = [f"{c}\n{int(T)}°C\n{l:.1f} wt%" for c,T,l in zip(dfp.Catalyst, dfp.T, dfp.Load)]

# Normalize each KPI to 0–1 (robust min-max)
def minmax(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

O2n  = minmax(dfp["Oxygen"])
H2On = minmax(dfp["Water"])
Vn   = minmax(dfp["Viscosity"])
DODn = minmax(dfp["DOD"])
HHVn = minmax(dfp["HHV"])

# Score: smaller is better
# (Tune weights if needed)
score = 0.35*O2n + 0.25*H2On + 0.15*Vn - 0.15*DODn - 0.10*HHVn
best_i = int(np.argmin(score))

best_i, dfp.loc[best_i, ["Catalyst","T","Load","Oxygen","Water","HHV","Viscosity","DOD"]]


In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(dfp))

plt.figure(figsize=(12,5))

# Green shades (benefits)
plt.bar(x, dfp["DOD"], color="#2ca25f", label="DOD (%)")
plt.bar(x, dfp["HHV"], bottom=dfp["DOD"], color="#66c2a4", label="HHV (MJ/kg)")

# Blue shades (penalties)
bottom_pen = dfp["DOD"] + dfp["HHV"]
plt.bar(x, dfp["Oxygen"], bottom=bottom_pen, color="#3182bd", label="Oxygen wt%")
plt.bar(x, dfp["Water"], bottom=bottom_pen + dfp["Oxygen"], color="#6baed6", label="Water wt%")
plt.bar(x, dfp["Viscosity"], bottom=bottom_pen + dfp["Oxygen"] + dfp["Water"],
        color="#9ecae1", label="Viscosity (cP)")

# Highlight best
plt.scatter([best_i], [bottom_pen.iloc[best_i] + dfp["Oxygen"].iloc[best_i] + dfp["Water"].iloc[best_i] + dfp["Viscosity"].iloc[best_i]],
            s=180, marker="*", color="black", label="Best-compromise")

plt.xticks(x, labels)
plt.ylabel("Stacked KPI (raw units mixed)")
plt.title("Pareto Points – RAW Stacked KPIs (green=benefits, blue=penalties)")
plt.legend(ncol=3, frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
!apt-get -qq install graphviz
!pip -q install graphviz

from graphviz import Digraph

dot = Digraph("PID_Flow", format="png")
dot.attr(rankdir="TB", bgcolor="white")

# Nodes
dot.node("S", "START", shape="oval", style="filled", fillcolor="black", fontcolor="white")
dot.node("A", "P&ID Source (DWG)\n• Original engineering drawing\n• Human readable", shape="box", style="rounded,filled", fillcolor="#e3f2fd")
dot.node("B", "Convert DWG → DXF\n• Export CAD geometry\n• Entities: LINE/POLYLINE/INSERT/TEXT", shape="box", style="rounded,filled", fillcolor="#e3f2fd")
dot.node("C", "Parse DXF (ezdxf)\nExtract:\n• Pipes (LINE/POLYLINE)\n• Equip symbols (INSERT)\n• Tags (TEXT)\n• Layers = process/signal", shape="box", style="rounded,filled", fillcolor="#fff3e0")
dot.node("D", "Build P&ID JSON\nnodes[] + edges[]\n• node: id,type,(x,y)\n• edge: from,to,type", shape="box", style="rounded,filled", fillcolor="#fff3e0")
dot.node("E", "Normalize + Clean Geometry\n• remove outliers\n• rescale coords", shape="box", style="rounded,filled", fillcolor="#f3e5f5")
dot.node("F", "KNN Connectivity (sanity)\n• local edges only\n• remove long-range false links", shape="box", style="rounded,filled", fillcolor="#f3e5f5")
dot.node("G", "ML-ready Graph\nG=(V,E)\n• nodes: equipment/reactors/instruments\n• edges: process pipes + signal lines", shape="box", style="rounded,filled", fillcolor="#e0f7fa")
dot.node("H", "GNN + pyMOO\n• learn KPI surrogate\n• Pareto-optimal setpoints", shape="box", style="rounded,filled", fillcolor="#e0f7fa")
dot.node("T", "STOP", shape="oval", style="filled", fillcolor="black", fontcolor="white")

# Edges
dot.edge("S","A")
dot.edge("A","B")
dot.edge("B","C")
dot.edge("C","D")
dot.edge("D","E")
dot.edge("E","F")
dot.edge("F","G")
dot.edge("G","H")
dot.edge("H","T")

dot

In [ ]:
from google.colab import files

# Render the dot object to a PNG file
dot.render('PID_Flow_Workflow', format='png', view=False)

# Download the generated PNG file
files.download('PID_Flow_Workflow.png')